# Agent 2 — Competitor Intelligence AI Agent


**Goal:** Given an industry/category keyword, find existing companies in that space,
group them with **K-Means clustering** (e.g. Leaders / Challengers / Niche players),
and use a **Random Forest** to rank which features (funding, rounds, milestones,
relationships, age) drive a company's overall strength score.

**Dataset used:** `data/startup_objects.csv` (Crunchbase companies)

**Output:** Competitor table (company name,country,market position(Market Leader/Challenger/Niche Player),position explanation,strengths,weaknesses )
returned by `agent_competitor(keyword)`. It gives top 5 competitors.



## 1. Imports & load data

In [21]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore")

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestRegressor

DATA_DIR = "../../data"

cols = [
    "id",
    "entity_type",
    "name",
    "category_code",
    "status",
    "country_code",
    "founded_at",
    "funding_total_usd",
    "funding_rounds",
    "milestones",
    "relationships"
]

companies = pd.read_csv(
    f"{DATA_DIR}/startup_objects.csv",
    usecols=cols,
    low_memory=False
)

companies = companies[
    companies["entity_type"] == "Company"
].copy()

print("Companies:", len(companies))

companies.head()


Companies: 196553


,id,entity_type,name,category_code,status,founded_at,country_code,funding_rounds,funding_total_usd,milestones,relationships
0,c:1,Company,Wetpaint,web,operating,2005-10-17,USA,3,39750000.0,5,17
1,c:10,Company,Flektor,games_video,acquired,NaN,USA,0,0.0,0,6
2,c:100,Company,There,games_video,acquired,NaN,USA,0,0.0,4,12
3,c:10000,Company,MYWEBBO,network_hosting,operating,2008-07-26,NaN,0,0.0,0,0
4,c:10001,Company,THE Movie Streamer,games_video,operating,2008-07-26,NaN,0,0.0,0,0


## 2. Dataset Cleaning

In [22]:
numeric_cols = [
    "funding_total_usd",
    "funding_rounds",
    "milestones",
    "relationships"
]

for col in numeric_cols:
    companies[col] = pd.to_numeric(
        companies[col],
        errors="coerce"
    ).fillna(0)

companies["category_code"] = (
    companies["category_code"]
    .fillna("unknown")
)

companies["founded_at"] = pd.to_datetime(
    companies["founded_at"],
    errors="coerce"
)

companies.head()

,id,entity_type,name,category_code,status,founded_at,country_code,funding_rounds,funding_total_usd,milestones,relationships
0,c:1,Company,Wetpaint,web,operating,2005-10-17,USA,3,39750000.0,5,17
1,c:10,Company,Flektor,games_video,acquired,NaT,USA,0,0.0,0,6
2,c:100,Company,There,games_video,acquired,NaT,USA,0,0.0,4,12
3,c:10000,Company,MYWEBBO,network_hosting,operating,2008-07-26,NaN,0,0.0,0,0
4,c:10001,Company,THE Movie Streamer,games_video,operating,2008-07-26,NaN,0,0.0,0,0


## 3. Feature engineering


In [23]:
companies["company_age_years"] = (
    pd.Timestamp("2014-01-01")
    - companies["founded_at"]
).dt.days / 365.25

companies["company_age_years"] = (
    companies["company_age_years"]
    .clip(lower=0)
    .fillna(0)
)

companies[["company_age_years"]].head()

,company_age_years
0,8.208077
1,0.000000
2,0.000000
3,5.434634
4,5.434634


## 4. Random Forest - Find Company Strength Pattern


In [24]:
FEATURES = [
    "funding_total_usd",
    "funding_rounds",
    "milestones",
    "relationships",
    "company_age_years"
]

train_features = [
    "funding_rounds",
    "milestones",
    "relationships",
    "company_age_years"
]

X = companies[train_features]
y = companies["funding_total_usd"]

rf = RandomForestRegressor(
    n_estimators=100,
    max_depth=8,
    random_state=42,
    n_jobs=-1
)

rf.fit(X, y)

feature_importance = pd.Series(
    rf.feature_importances_,
    index=train_features
).sort_values(ascending=False)

print("Factors affecting company strength")

feature_importance


Factors affecting company strength


relationships        0.361599
company_age_years    0.323410
funding_rounds       0.253256
milestones           0.061735
dtype: float64

## 5. Predicts Strength Score

In [25]:
companies["strength_score"] = rf.predict(X)

companies[
    [
        "name",
        "strength_score"
    ]
].head()

,name,strength_score
0,Wetpaint,3.582067e+07
1,Flektor,0.000000e+00
2,There,0.000000e+00
3,MYWEBBO,0.000000e+00
4,THE Movie Streamer,0.000000e+00


## 6. K-Means clustering — group competitors into segments


In [26]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(
    companies[FEATURES]
)

kmeans = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=10
)

companies["cluster"] = kmeans.fit_predict(
    X_scaled
)

companies.head()

,id,entity_type,name,category_code,status,founded_at,country_code,funding_rounds,funding_total_usd,milestones,relationships,company_age_years,strength_score,cluster
0,c:1,Company,Wetpaint,web,operating,2005-10-17,USA,3,39750000.0,5,17,8.208077,3.582067e+07,1
1,c:10,Company,Flektor,games_video,acquired,NaT,USA,0,0.0,0,6,0.000000,0.000000e+00,0
2,c:100,Company,There,games_video,acquired,NaT,USA,0,0.0,4,12,0.000000,0.000000e+00,2
3,c:10000,Company,MYWEBBO,network_hosting,operating,2008-07-26,NaN,0,0.0,0,0,5.434634,0.000000e+00,0
4,c:10001,Company,THE Movie Streamer,games_video,operating,2008-07-26,NaN,0,0.0,0,0,5.434634,0.000000e+00,0


## 7. Converts Clusters into Business Positions

In [27]:
cluster_rank = (
    companies
    .groupby("cluster")["strength_score"]
    .mean()
    .sort_values(ascending=False)
)

label_map = {
    cluster_rank.index[0]: "Market Leader",
    cluster_rank.index[1]: "Challenger",
    cluster_rank.index[2]: "Niche Player"
}

companies["market_position"] = (
    companies["cluster"]
    .map(label_map)
)

companies["market_position"].value_counts()

market_position
Challenger       103560
Niche Player      83086
Market Leader      9907
Name: count, dtype: int64

## 8. Keywords Category Matching

In [28]:
ALL_CATEGORIES = companies["category_code"].unique().tolist()

KEYWORD_TO_CATEGORY = {
    "food": "hospitality",
    "delivery": "hospitality",
    "restaurant": "hospitality",
    "finance": "finance",
    "fintech": "finance",
    "health": "health",
    "fitness": "health",
    "education": "education",
    "learning": "education",
    "software": "software",
    "app": "mobile",
    "mobile": "mobile",
    "game": "games_video",
    "travel": "travel",
    "shopping": "ecommerce",
    "ecommerce": "ecommerce"
}

def match_category(keyword):
    keyword = keyword.lower()
    for key, cat in KEYWORD_TO_CATEGORY.items():
        if key in keyword and cat in ALL_CATEGORIES:
            return cat
    return "software"

## 9. Generates Business Explanation

In [36]:
def generate_business_insight(row):
    strengths = []
    weaknesses = []

    category = row["category_code"]
    subset = companies[companies["category_code"] == category]

    def normalize(col):
        max_val = subset[col].max()
        min_val = subset[col].min()
        if max_val == min_val:
            return 0.5
        return (row[col] - min_val) / (max_val - min_val)

    funding_score = normalize("funding_total_usd")
    rounds_score = normalize("funding_rounds")
    rel_score = normalize("relationships")

    # ------------------------
    # Balanced SWOT logic
    # ------------------------

    def evaluate(metric, name):
        if metric >= 0.75:
            strengths.append(f"Strong {name} compared to competitors")
        elif metric <= 0.35:
            weaknesses.append(f"Weak {name} compared to top competitors")
        else:
            strengths.append(f"Average {name} performance")

    evaluate(funding_score, "financial strength")
    evaluate(rounds_score, "funding activity")
    evaluate(rel_score, "business network")

    position_explanation = {
        "Market Leader": "Dominates its segment with strong overall competitive metrics",
        "Challenger": "Strong performer with potential to overtake leaders",
        "Niche Player": "Focused player serving a specialized market segment"
    }

    return {
        "market_position": row["market_position"],
        "position_explanation": position_explanation[row["market_position"]],
        "strengths": strengths,
        "weaknesses": weaknesses
    }

## 10. agent_competitor(keyword)

In [37]:
def agent_competitor(keyword: str, top_n=5):
    category = match_category(keyword)

    subset = companies[
        companies["category_code"] == category
    ]

    top_companies = (
        subset
        .sort_values(
            "strength_score",
            ascending=False
        )
        .head(top_n)
    )

    competitors = []

    for _, row in top_companies.iterrows():
        insight = generate_business_insight(row)

        competitors.append({
            "company_name": row["name"],
            "country": row["country_code"],
            "market_position": insight["market_position"],
            "position_explanation": insight["position_explanation"],
            "strengths": insight["strengths"],
            "weaknesses": insight["weaknesses"]
        })

    return {
        "agent": "Competitor Intelligence Agent",
        "input_keyword": keyword,
        "matched_category": category,
        "num_companies_analyzed": int(len(subset)),
        "competitors": competitors
    }

## 11. Test the agent

In [38]:
import json

result = agent_competitor("food delivery")

print(
    json.dumps(
        result,
        indent=2,
        default=str
    )
)


{
  "agent": "Competitor Intelligence Agent",
  "input_keyword": "food delivery",
  "matched_category": "hospitality",
  "num_companies_analyzed": 768,
  "competitors": [
    {
      "company_name": "GrubHub",
      "country": "USA",
      "market_position": "Market Leader",
      "position_explanation": "Dominates its segment with strong overall competitive metrics",
      "strengths": [
        "Strong funding activity compared to competitors"
      ],
      "weaknesses": [
        "Weak financial strength compared to top competitors",
        "Weak business network compared to top competitors"
      ]
    },
    {
      "company_name": "Lot18",
      "country": "USA",
      "market_position": "Market Leader",
      "position_explanation": "Dominates its segment with strong overall competitive metrics",
      "strengths": [
        "Strong funding activity compared to competitors",
        "Average business network performance"
      ],
      "weaknesses": [
        "Weak financial s

## 7. Next steps
- Move the feature engineering, RF strength model and K-Means clustering into
  `agent2_competitor/clustering.py`, and expose `agent_competitor()` via
  `agent2_competitor/competitor_agent.py`.
- Cache the fitted `rf`, `kmeans`, `scaler`, and the processed `companies`
  DataFrame at startup (pickle or joblib) so the FastAPI endpoint doesn't
  retrain on every request.
- Add pricing/rating columns if you enrich the dataset with scraped data later.
